# Studi Kasus HCI & Data Visualization: Dashboard Layanan Perpustakaan Kampus


## 1. Tujuan dan kaitan dengan HCI

- **Hierarki visual:** KPI penting ditempatkan di bagian atas.
- **Konsistensi:** kartu, warna, label, dan pola interaksi digunakan secara konsisten.
- **Umpan balik:** tombol filter memperbarui KPI, grafik, dan tabel.
- **Desain responsif:** tata letak menyesuaikan layar desktop maupun ponsel.
- **Aksesibilitas dasar:** label input, fokus keyboard, dan deskripsi grafik disediakan.

Prinsip ini merupakan rancangan antarmuka yang diterapkan dalam studi kasus; evaluasi formal dengan pengguna tetap perlu dilakukan terpisah.

In [1]:
# 2. Import library dan siapkan data dummy
import json
from pathlib import Path
from datetime import date
import numpy as np
import pandas as pd
from IPython.display import display, HTML, FileLink

np.random.seed(42)
N = 800
dates = pd.date_range('2026-01-01', periods=90, freq='D')
services = ['Peminjaman Buku', 'Pengembalian', 'Keanggotaan', 'Referensi', 'Ruang Diskusi']
faculties = ['Teknik', 'Ekonomi', 'Hukum', 'Kesehatan', 'Ilmu Komputer']
time_slots = ['08.00–10.00', '10.00–12.00', '12.00–14.00', '14.00–16.00']
service = np.random.choice(services, N, p=[.31,.22,.14,.18,.15])
wait = np.clip(np.random.gamma(2.1, 4.0, N) + np.where(service=='Peminjaman Buku',2,0), 1, 35).round().astype(int)
satisfaction = np.clip(np.round(5 - wait/13 + np.random.normal(0,.65,N)), 1, 5).astype(int)
df = pd.DataFrame({
    'date': np.random.choice(dates, N),
    'service': service,
    'faculty': np.random.choice(faculties, N),
    'time_slot': np.random.choice(time_slots, N, p=[.25,.34,.23,.18]),
    'wait': wait,
    'duration': np.clip(np.random.normal(8, 3, N) + np.where(service=='Ruang Diskusi',5,0), 2, 25).round().astype(int),
    'satisfaction': satisfaction
})
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')
df = df.sort_values(['date','service']).reset_index(drop=True)
print(f'Jumlah catatan: {len(df)}')
display(df.head())

Jumlah catatan: 800


,date,service,faculty,time_slot,wait,duration,satisfaction
0,2026-01-01,Peminjaman Buku,Teknik,12.00–14.00,9,12,5
1,2026-01-01,Peminjaman Buku,Ilmu Komputer,08.00–10.00,9,4,4
2,2026-01-01,Peminjaman Buku,Teknik,12.00–14.00,8,11,4
3,2026-01-01,Peminjaman Buku,Hukum,14.00–16.00,25,5,2
4,2026-01-01,Peminjaman Buku,Kesehatan,14.00–16.00,8,9,4


## 3. Ringkasan awal data

Ringkasan ini membantu memeriksa data sebelum divisualisasikan.

In [2]:
summary = pd.DataFrame({
    'Metrik': ['Total kunjungan', 'Rata-rata waktu tunggu (menit)', 'Rata-rata durasi layanan (menit)', 'Rata-rata kepuasan (skala 1–5)'],
    'Nilai': [len(df), round(df['wait'].mean(), 2), round(df['duration'].mean(), 2), round(df['satisfaction'].mean(), 2)]
})
display(summary)
display(df['service'].value_counts().rename_axis('Jenis layanan').to_frame('Jumlah kunjungan'))

,Metrik,Nilai
0,Total kunjungan,800.00
1,Rata-rata waktu tunggu (menit),9.30
2,Rata-rata durasi layanan (menit),8.78
3,Rata-rata kepuasan (skala 1–5),4.23


,Jumlah kunjungan
Jenis layanan,
Peminjaman Buku,252
Pengembalian,170
Referensi,133
Ruang Diskusi,127
Keanggotaan,118


## 4. Buat dashboard HTML + CSS + JavaScript


In [3]:
# Tempel template dashboard yang panjang di sel ini.
# File HTML juga disediakan terpisah di repositori GitHub.
# Jalankan sel berikut untuk membentuk dashboard dari template yang disimpan sebagai string.
HTML_TEMPLATE = r'''<!DOCTYPE html>
<html lang="id">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>LibraryPulse | Dashboard Layanan Perpustakaan</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
:root{--bg:#f4f7fb;--card:#fff;--ink:#17243a;--muted:#6d7b90;--blue:#3659d9;--teal:#16a394;--orange:#f0a43c;--line:#e6ebf2;--shadow:0 8px 26px rgba(29,48,83,.07)}
*{box-sizing:border-box}body{margin:0;background:var(--bg);font-family:Inter,Segoe UI,Arial,sans-serif;color:var(--ink)}
header{background:linear-gradient(120deg,#172c67,#3659d9);color:white;padding:28px clamp(18px,4vw,56px) 30px}
.topline{display:flex;justify-content:space-between;align-items:center;gap:16px;flex-wrap:wrap}.brand{font-size:13px;font-weight:800;letter-spacing:2px;opacity:.85}.pill{border:1px solid rgba(255,255,255,.35);border-radius:999px;padding:7px 12px;font-size:12px}
h1{font-size:clamp(25px,3vw,36px);margin:18px 0 8px}.subtitle{max-width:760px;line-height:1.6;color:#e2e9ff;margin:0}
main{max-width:1400px;margin:auto;padding:24px clamp(14px,3vw,40px) 48px}.filters{display:flex;gap:14px;align-items:end;flex-wrap:wrap;background:var(--card);padding:16px 18px;border-radius:16px;box-shadow:var(--shadow);margin-bottom:18px}
label{display:block;color:var(--muted);font-size:12px;font-weight:700;margin-bottom:7px}select,input{border:1px solid #d8e0eb;background:#fff;border-radius:9px;padding:10px 12px;color:var(--ink);font:inherit;min-width:170px}
button{border:0;background:var(--blue);color:white;border-radius:9px;padding:11px 15px;font-weight:700;cursor:pointer}button.secondary{background:#edf1ff;color:var(--blue)}button:focus,select:focus,input:focus{outline:3px solid #b9c7ff;outline-offset:2px}
.kpis{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:14px;margin-bottom:18px}.kpi,.panel{background:var(--card);border-radius:16px;box-shadow:var(--shadow);border:1px solid rgba(230,235,242,.7)}
.kpi{padding:18px 20px;min-height:125px}.kpi .label{color:var(--muted);font-size:13px;font-weight:700}.kpi .value{font-size:clamp(25px,2.5vw,32px);font-weight:800;margin:10px 0 5px;letter-spacing:-.8px}.kpi .hint{font-size:12px;color:var(--muted)}
.layout{display:grid;grid-template-columns:1.15fr 1fr;gap:16px;margin-bottom:16px}.panel{padding:18px;min-width:0}.panel h2{font-size:16px;margin:0 0 4px}.panel p{font-size:12px;color:var(--muted);margin:0 0 12px;line-height:1.5}.chart{height:310px;width:100%}
.table-panel{margin-top:16px}.table-wrap{overflow:auto;margin-top:10px}table{width:100%;border-collapse:collapse;font-size:13px;white-space:nowrap}th{text-align:left;color:var(--muted);font-size:11px;text-transform:uppercase;letter-spacing:.6px;background:#f7f9fc}th,td{padding:12px 13px;border-bottom:1px solid var(--line)}tbody tr:hover{background:#f7f9ff}.tag{display:inline-block;padding:4px 8px;border-radius:999px;background:#e5f7f3;color:#087e72;font-size:11px;font-weight:800}.empty{text-align:center;color:var(--muted);padding:30px}
footer{max-width:1400px;margin:auto;padding:0 24px 28px;color:var(--muted);font-size:12px;line-height:1.6}.footgrid{display:grid;grid-template-columns:1fr 1fr;gap:16px}.note{background:#eaf0ff;border-left:4px solid var(--blue);padding:14px 16px;border-radius:10px;color:#33436a}
@media(max-width:850px){.kpis{grid-template-columns:repeat(2,minmax(0,1fr))}.layout{grid-template-columns:1fr}.chart{height:290px}.footgrid{grid-template-columns:1fr}}
@media(max-width:480px){.kpis{grid-template-columns:1fr 1fr;gap:9px}.kpi{padding:14px;min-height:115px}.kpi .value{font-size:23px}select,input{min-width:140px;width:100%}.filters>div{flex:1;min-width:130px}.panel{padding:13px}}
</style>
</head>
<body>
<header>
  <div class="topline"><div class="brand">LIBRARYPULSE · HUMAN-CENTERED DATA</div><div class="pill">Studi Kasus HCI & Data Visualization</div></div>
  <h1>Dashboard Layanan Perpustakaan Kampus</h1>
  <p class="subtitle">Ringkasan kunjungan, penggunaan layanan, waktu tunggu, dan kepuasan mahasiswa. Filter interaktif membantu pengguna menemukan informasi yang relevan tanpa harus membaca seluruh data.</p>
</header>
<main>
  <section class="filters" aria-label="Filter dashboard">
    <div><label for="serviceFilter">Jenis layanan</label><select id="serviceFilter"><option value="Semua">Semua layanan</option></select></div>
    <div><label for="facultyFilter">Fakultas</label><select id="facultyFilter"><option value="Semua">Semua fakultas</option></select></div>
    <div><label for="fromDate">Dari tanggal</label><input id="fromDate" type="date"></div>
    <div><label for="toDate">Sampai tanggal</label><input id="toDate" type="date"></div>
    <div><button id="applyBtn">Terapkan filter</button></div><div><button id="resetBtn" class="secondary">Reset</button></div>
  </section>
  <section class="kpis" aria-label="Indikator utama">
    <div class="kpi"><div class="label">Total kunjungan</div><div class="value" id="kVisits">—</div><div class="hint">Jumlah catatan sesuai filter</div></div>
    <div class="kpi"><div class="label">Rata-rata waktu tunggu</div><div class="value" id="kWait">—</div><div class="hint">Menit sebelum dilayani</div></div>
    <div class="kpi"><div class="label">Kepuasan rata-rata</div><div class="value" id="kSat">—</div><div class="hint">Skala penilaian 1–5</div></div>
    <div class="kpi"><div class="label">Layanan paling digunakan</div><div class="value" id="kPopular" style="font-size:21px">—</div><div class="hint">Berdasarkan jumlah kunjungan</div></div>
  </section>
  <section class="layout">
    <article class="panel"><h2>Tren kunjungan harian</h2><p>Melihat perubahan jumlah kunjungan dari waktu ke waktu.</p><div id="trendChart" class="chart" role="img" aria-label="Grafik garis tren kunjungan harian"></div></article>
    <article class="panel"><h2>Penggunaan tiap layanan</h2><p>Membandingkan volume kunjungan antarjenis layanan.</p><div id="serviceChart" class="chart" role="img" aria-label="Grafik batang penggunaan layanan"></div></article>
  </section>
  <section class="layout">
    <article class="panel"><h2>Waktu tunggu dan kepuasan</h2><p>Setiap titik mewakili satu kunjungan. Grafik membantu melihat pola, bukan membuktikan sebab-akibat.</p><div id="scatterChart" class="chart" role="img" aria-label="Grafik sebar waktu tunggu dan kepuasan"></div></article>
    <article class="panel"><h2>Rata-rata durasi layanan</h2><p>Durasi rata-rata dalam menit untuk tiap jenis layanan.</p><div id="durationChart" class="chart" role="img" aria-label="Grafik durasi layanan"></div></article>
  </section>
  <section class="panel table-panel"><h2>Rincian kunjungan</h2><p>Menampilkan maksimal 100 catatan pertama setelah filter diterapkan.</p><div class="table-wrap"><table><thead><tr><th>Tanggal</th><th>Layanan</th><th>Fakultas</th><th>Waktu kunjungan</th><th>Waktu tunggu</th><th>Durasi</th><th>Kepuasan</th></tr></thead><tbody id="dataTable"></tbody></table></div><div id="tableInfo" style="font-size:12px;color:var(--muted);margin-top:12px"></div></section>
</main>
<footer><div class="footgrid"><div class="note"><strong>Catatan data:</strong> seluruh data dalam dashboard ini adalah data sintetis/dummy untuk demonstrasi tugas, bukan data resmi perpustakaan atau hasil survei nyata.</div><div><strong>Kaitan dengan HCI:</strong> hierarki visual, konsistensi komponen, label filter yang jelas, umpan balik langsung, desain responsif, serta kontras dan fokus keyboard diterapkan untuk membantu pengguna memahami informasi secara efisien.</div></div></footer>
<script>
const DATA = __DATA__;
const fmt = n => new Intl.NumberFormat('id-ID').format(n);
const avg = arr => arr.length ? arr.reduce((a,b)=>a+b,0)/arr.length : 0;
const services = [...new Set(DATA.map(d=>d.service))].sort();
const faculties = [...new Set(DATA.map(d=>d.faculty))].sort();
const serviceSelect=document.getElementById('serviceFilter'), facultySelect=document.getElementById('facultyFilter');
services.forEach(v=>serviceSelect.add(new Option(v,v))); faculties.forEach(v=>facultySelect.add(new Option(v,v)));
const dates=DATA.map(d=>d.date).sort();
document.getElementById('fromDate').value=dates[0]||'';document.getElementById('toDate').value=dates[dates.length-1]||'';
function getFiltered(){const s=serviceSelect.value,f=facultySelect.value,from=document.getElementById('fromDate').value,to=document.getElementById('toDate').value;
 return DATA.filter(d=>(s==='Semua'||d.service===s)&&(f==='Semua'||d.faculty===f)&&(!from||d.date>=from)&&(!to||d.date<=to));}
const baseLayout={margin:{l:48,r:18,t:10,b:45},paper_bgcolor:'rgba(0,0,0,0)',plot_bgcolor:'rgba(0,0,0,0)',font:{family:'Inter, Segoe UI, Arial',color:'#6d7b90',size:11},xaxis:{gridcolor:'#edf1f6',zerolinecolor:'#edf1f6'},yaxis:{gridcolor:'#edf1f6',zerolinecolor:'#edf1f6'},showlegend:false};
function draw(){const d=getFiltered();document.getElementById('kVisits').textContent=fmt(d.length);document.getElementById('kWait').textContent=d.length?avg(d.map(x=>x.wait)).toFixed(1)+' mnt':'—';document.getElementById('kSat').textContent=d.length?avg(d.map(x=>x.satisfaction)).toFixed(2)+' / 5':'—';
 const counts={};d.forEach(x=>counts[x.service]=(counts[x.service]||0)+1);const popular=Object.entries(counts).sort((a,b)=>b[1]-a[1])[0];document.getElementById('kPopular').textContent=popular?popular[0]:'—';
 const daily={};d.forEach(x=>daily[x.date]=(daily[x.date]||0)+1);const dayKeys=Object.keys(daily).sort();
 Plotly.react('trendChart',[{x:dayKeys,y:dayKeys.map(k=>daily[k]),type:'scatter',mode:'lines+markers',line:{color:'#3659d9',width:3},marker:{size:6,color:'#3659d9'},hovertemplate:'%{x}<br>Kunjungan: %{y}<extra></extra>'}],{...baseLayout,yaxis:{...baseLayout.yaxis,title:'Kunjungan'},xaxis:{...baseLayout.xaxis,title:'Tanggal',tickangle:-25}}, {responsive:true,displayModeBar:false});
 const sk=Object.keys(counts).sort((a,b)=>counts[b]-counts[a]);Plotly.react('serviceChart',[{x:sk,y:sk.map(k=>counts[k]),type:'bar',marker:{color:['#3659d9','#16a394','#f0a43c','#7c6ee6','#52a5d8']},hovertemplate:'%{x}<br>%{y} kunjungan<extra></extra>'}],{...baseLayout,xaxis:{...baseLayout.xaxis,tickangle:-20},yaxis:{...baseLayout.yaxis,title:'Kunjungan'}},{responsive:true,displayModeBar:false});
 Plotly.react('scatterChart',[{x:d.map(x=>x.wait),y:d.map(x=>x.satisfaction),type:'scatter',mode:'markers',text:d.map(x=>x.service),marker:{size:8,opacity:.58,color:'#16a394'},hovertemplate:'Layanan: %{text}<br>Waktu tunggu: %{x} menit<br>Kepuasan: %{y}/5<extra></extra>'}],{...baseLayout,xaxis:{...baseLayout.xaxis,title:'Waktu tunggu (menit)'},yaxis:{...baseLayout.yaxis,title:'Kepuasan (1–5)',range:[0.5,5.5],dtick:1}},{responsive:true,displayModeBar:false});
 const durations={};d.forEach(x=>{(durations[x.service]??=[]).push(x.duration)});const dk=Object.keys(durations).sort();Plotly.react('durationChart',[{x:dk,y:dk.map(k=>avg(durations[k])),type:'bar',marker:{color:'#16a394'},hovertemplate:'%{x}<br>Rata-rata %{y:.1f} menit<extra></extra>'}],{...baseLayout,xaxis:{...baseLayout.xaxis,tickangle:-20},yaxis:{...baseLayout.yaxis,title:'Menit'}},{responsive:true,displayModeBar:false});
 const tbody=document.getElementById('dataTable');tbody.innerHTML='';d.slice(0,100).forEach(x=>{const tr=document.createElement('tr');[x.date,x.service,x.faculty,x.time_slot,x.wait+' mnt',x.duration+' mnt'].forEach(v=>{const td=document.createElement('td');td.textContent=v;tr.appendChild(td)});const td=document.createElement('td');td.innerHTML='<span class="tag">'+x.satisfaction+'/5</span>';tr.appendChild(td);tbody.appendChild(tr)});if(!d.length)tbody.innerHTML='<tr><td colspan="7" class="empty">Tidak ada data yang cocok dengan filter.</td></tr>';document.getElementById('tableInfo').textContent=`Menampilkan ${Math.min(d.length,100)} dari ${d.length} catatan yang cocok.`;
}
document.getElementById('applyBtn').addEventListener('click',draw);document.getElementById('resetBtn').addEventListener('click',()=>{serviceSelect.value='Semua';facultySelect.value='Semua';document.getElementById('fromDate').value=dates[0]||'';document.getElementById('toDate').value=dates[dates.length-1]||'';draw();});
draw();
</script>
</body></html>'''

records = df.to_dict(orient='records')
records_json = json.dumps(records, ensure_ascii=False)
html_content = HTML_TEMPLATE.replace('__DATA__', records_json)
output_dir = Path('/content/studi_kasus_hci_visualisasi_perpustakaan')
output_dir.mkdir(parents=True, exist_ok=True)
html_path = output_dir / 'dashboard_perpustakaan.html'
html_path.write_text(html_content, encoding='utf-8')
print('Dashboard berhasil dibuat:', html_path)
print('Ukuran file:', round(html_path.stat().st_size / 1024, 1), 'KB')

Dashboard berhasil dibuat: /content/studi_kasus_hci_visualisasi_perpustakaan/dashboard_perpustakaan.html
Ukuran file: 129.2 KB


## 5. Preview dan unduh file HTML


In [4]:
# Coba tampilkan preview inline (beberapa konfigurasi Colab dapat membatasi JavaScript)
display(HTML('<p>File dashboard dibuat. Gunakan tautan unduhan di bawah, lalu buka file HTML di browser.</p>'))
display(FileLink(str(html_path)))

# Untuk mengunduh langsung dari Google Colab, jalankan:
from google.colab import files
files.download(str(html_path))

/content/studi_kasus_hci_visualisasi_perpustakaan/dashboard_perpustakaan.html

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Ekspor data dummy ke CSV


In [5]:
csv_path = output_dir / 'data_dummy_perpustakaan.csv'
df.to_csv(csv_path, index=False)
print('CSV tersimpan:', csv_path)
# Di Colab, aktifkan baris berikut jika ingin mengunduh CSV:
# files.download(str(csv_path))

CSV tersimpan: /content/studi_kasus_hci_visualisasi_perpustakaan/data_dummy_perpustakaan.csv


## 7. Pemeriksaan sederhana

Pemeriksaan ini memastikan tidak ada nilai kosong pada kolom inti dan nilai kepuasan berada dalam skala 1–5.

In [6]:
required_cols = ['date','service','faculty','time_slot','wait','duration','satisfaction']
assert df[required_cols].notna().all().all(), 'Ada nilai kosong pada data inti.'
assert df['satisfaction'].between(1,5).all(), 'Nilai kepuasan harus berada pada skala 1–5.'
assert (df['wait'] >= 0).all() and (df['duration'] > 0).all(), 'Waktu harus bernilai positif.'
print('Semua pemeriksaan data dasar berhasil.')

Semua pemeriksaan data dasar berhasil.
